In [ ]:
from pathlib import Path
import pandas as pd

from robovast.common.campaign_data import (list_config_dirs, list_run_dirs,
                                           read_test_result)

DATA_DIR = ''

def aggregate_config(config_dir):
    """One aggregated row for a config: mean per-run metrics + failure_rate (test.xml)."""
    config_dir = Path(config_dir)
    runs = list_run_dirs(config_dir)
    rows = [pd.read_csv(r / 'metrics.csv').iloc[0]
            for r in runs if (r / 'metrics.csv').exists()]
    rec = pd.DataFrame(rows).mean(numeric_only=True).to_dict() if rows else {}
    failures = 0
    for r in runs:
        try:
            if not read_test_result(r)['success']:
                failures += 1
        except FileNotFoundError:
            failures += 1
    rec['failure_rate'] = failures / len(runs) if runs else 0.0
    rec['n_runs'] = len(runs)
    return rec

rows = []
for cfg in list_config_dirs(Path(DATA_DIR)):
    rec = aggregate_config(cfg)
    rec['config'] = cfg.name
    rows.append(rec)
metrics = pd.DataFrame(rows)
print(f'{len(metrics)} config(s)')
metrics

In [ ]:
# Search outcome: how/why this campaign stopped (search mode only). Reads the
# parseable stop record persisted on the campaign row; silently skipped for batch
# campaigns or older stores without the columns.
import os as _os, glob as _glob, sqlite3 as _sqlite3
_db = _os.path.join(DATA_DIR, 'campaign.db')
if not _os.path.exists(_db):
    _hits = _glob.glob(_os.path.join(DATA_DIR or '.', '**', 'campaign.db'), recursive=True)
    _db = _hits[0] if _hits else _db
try:
    _con = _sqlite3.connect(_db)
    _row = _con.execute('SELECT stop_kind, stop_reason, batches, elapsed_s '
                        'FROM campaign ORDER BY created_at DESC LIMIT 1').fetchone()
    _con.close()
except Exception:
    _row = None
if _row and _row[0]:
    _kind, _reason, _nb, _es = _row
    print(f'🏁 Search stopped: {_reason}')
    print(f'   {_nb} batch(es), {_es:.0f}s  (criterion: {_kind})')

In [ ]:
import matplotlib.pyplot as plt

# Per-config behavior + objective overview.
if len(metrics):
    measures = [c for c in ['max_tilt', 'drift_dist', 'landing_speed', 'control_effort']
                if c in metrics.columns]
    obj = 'failure_rate' if 'failure_rate' in metrics.columns else metrics.columns[0]
    if len(measures) >= 2:
        fig, ax = plt.subplots(figsize=(6, 5))
        sc = ax.scatter(metrics[measures[0]], metrics[measures[1]], c=metrics[obj], cmap='viridis')
        ax.set_xlabel(measures[0]); ax.set_ylabel(measures[1])
        fig.colorbar(sc, ax=ax, label=obj); ax.set_title('configs in behavior space')
        plt.tight_layout(); plt.show()
    display(metrics.sort_values(obj, ascending=False).head(10))
else:
    print('No configs found — run postprocessing to produce per-run metrics.csv.')